# Entropy Gate Pipeline — Colab runner

SPY and the 11 sector ETFs, 2016 onward. Before running anything, open the **key icon** in the left sidebar and add:

| Name | Value |
|---|---|
| `GITHUB_TOKEN` | A [personal access token](https://github.com/settings/tokens) with `repo` scope (the repo is private) |
| `APCA_API_KEY_ID` | Your Alpaca key ID |
| `APCA_API_SECRET_KEY` | Your Alpaca secret key |

Turn **Notebook access** on for each. No cell prints these values; they only go into a local `.env` that never leaves this runtime.

**Runtime → Change runtime type → T4 GPU** before running. Every step writes to Google Drive, so after a disconnect you can re-run from the top and finished work is skipped.

In [ ]:
import torch
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU attached: Runtime -> Change runtime type -> T4 GPU")

## 1. Mount Drive

Colab's disk is wiped when the runtime recycles. Downloaded bars (`data/`) and results (`outputs/`) live in **My Drive → entropy-gate-pipeline-data** instead.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/entropy-gate-pipeline-data'
for name in ('data', 'outputs'):
    os.makedirs(f'{DRIVE_DATA_DIR}/{name}', exist_ok=True)
print(f'persistent storage ready at {DRIVE_DATA_DIR}')

## 2. Get the latest code

In [ ]:
from google.colab import userdata

GITHUB_REPO = 'Gaire-commits/entropy-gate-pipeline'
REPO_DIR = '/content/entropy-gate-pipeline'
token = userdata.get('GITHUB_TOKEN')

if os.path.isdir(f'{REPO_DIR}/.git'):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone -q https://{token}@github.com/{GITHUB_REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

## 3. Storage, dependencies, credentials

In [ ]:
for name in ('data', 'outputs'):
    local_path, drive_path = f'{REPO_DIR}/{name}', f'{DRIVE_DATA_DIR}/{name}'
    if os.path.exists(local_path) and not os.path.islink(local_path):
        raise RuntimeError(f'{local_path} is a real folder, not a link to Drive; delete it and re-run')
    if not os.path.islink(local_path):
        os.symlink(drive_path, local_path)

!pip install -q -r requirements.txt

key, secret = userdata.get('APCA_API_KEY_ID'), userdata.get('APCA_API_SECRET_KEY')
with open('.env', 'w') as f:
    f.write(f'APCA_API_KEY_ID={key}\nAPCA_API_SECRET_KEY={secret}\nALPACA_DATA_FEED=iex\n')
del key, secret
print('ready: data/ and outputs/ point at Drive, .env written')

## 4. Check the pipeline

No credentials used. Unit tests, then synthetic data with planted signals: the gate has to pass trends and refuse zig-zags and tick noise, and every model has to recover a signal we know is there. If this fails, don't trust anything below it.

In [ ]:
!python -m pytest tests/ -q -p no:warnings
!python scripts/smoke_test.py 2>&1 | grep -v Warning

## 5. Download bars

SPY + 11 sector ETFs, 5-minute bars from 2016 (XLC starts in 2018). The first run takes a while; after that every symbol says `cached`. The last column is the share of 5-minute slots where IEX actually had a trade.

In [ ]:
!python scripts/fetch_data.py --config configs/etf_intraday.yaml

## 6. SPY: does the first half-hour predict the last half-hour?

The published intraday momentum effect, and a real-data check that the pipeline can find something known. Start here: this runs in minutes.

Each experiment runs three steps:
- **screen**: the gate's nightly readings
- **sweep**: every model and rule on every walk-forward fold, with 3 seeds each
- **summarize**: results with 95% intervals from resampling days

In [ ]:
CONFIG = 'configs/spy_gao.yaml'
!python scripts/run_screen.py --config {CONFIG}
!python scripts/sweep.py --config {CONFIG} 2>&1 | grep -v Warning
!python scripts/summarize.py --config {CONFIG}

## 7. The same question on all 12 ETFs

In [ ]:
CONFIG = 'configs/etf_gao.yaml'
!python scripts/run_screen.py --config {CONFIG}
!python scripts/sweep.py --config {CONFIG} 2>&1 | grep -v Warning
!python scripts/summarize.py --config {CONFIG}

## 8. ETFs, 4-hour window → 1-hour hold (slowest)

The deep models train on every fold with 3 seeds each, so this can take hours. If Colab disconnects, re-run sections 1–3 and this cell; finished folds are skipped.

The fast rules and logreg run first, so there's a first look at the results before the long part. Among the deep models, resnet2d is by far the slowest: it builds a 48×48 image for every channel of every sample. Each fold line ends with its duration in seconds, so you can judge from the first few folds whether to let it finish or stop it and summarize without it.

In [ ]:
CONFIG = 'configs/etf_intraday.yaml'
!python scripts/run_screen.py --config {CONFIG}
!python scripts/sweep.py --config {CONFIG} --archs always_up momentum_day momentum_window logreg 2>&1 | grep -v Warning
!python scripts/summarize.py --config {CONFIG}

In [ ]:
!python scripts/sweep.py --config {CONFIG} --archs cnn1d resnet1d inceptiontime resnet2d 2>&1 | grep -v Warning
!python scripts/summarize.py --config {CONFIG}

## 9. All results

Each experiment's tables are also saved on Drive under `outputs/<experiment>/`: `summary.md`, `overall.csv`, `gate.csv`.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
for name in ('spy_gao', 'etf_gao', 'etf_intraday'):
    path = Path('outputs') / name / 'summary.md'
    display(Markdown(path.read_text() if path.exists() else f'*{name}: not run yet*'))